# 01b — Contrôle qualité des métriques générées

À exécuter après le notebook `01_generer_csv_<corpus>.ipynb`. Le corpus est
sélectionné dans la cellule de configuration ; aucun CSV d'un autre corpus n'est
mélangé automatiquement.


## 1. Configuration


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from metric_registry import get_dataset
from pipeline_utils import detect_project_dir, metric_inventory, aggregate_metrics

PROJECT_DIR = detect_project_dir()
DATASET = "Mirabelle"  # "Mirabelle", "Nowledgeable" ou "Progsnap2"
JOIN_MODE = "outer"

SPEC = get_dataset(DATASET)
CSV_DIR = SPEC.csv_dir(PROJECT_DIR)
print("Corpus :", DATASET)
print("CSV    :", CSV_DIR)


## 2. Inventaire


In [ ]:
inventory_df = metric_inventory(CSV_DIR)
if inventory_df.empty:
    raise FileNotFoundError(f"Aucune métrique dans {CSV_DIR}. Exécutez d'abord le notebook 01 correspondant.")
display(inventory_df)


## 3. Agrégation par étudiant


In [ ]:
features, load_report = aggregate_metrics(CSV_DIR, join_mode=JOIN_MODE)
display(load_report)
print(f"Table agrégée : {features.shape[0]} étudiant(s) × {features.shape[1]-1} métrique(s)")
display(features.head())


## 4. Valeurs manquantes et distributions


In [ ]:
metric_cols = [c for c in features.columns if c != "SubjectID"]
for col in metric_cols:
    features[col] = pd.to_numeric(features[col], errors="coerce")

missing = (
    features[metric_cols].isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename("pct_manquant").reset_index().rename(columns={"index": "variable"})
)
display(missing)

display(features[metric_cols].describe().T)

plt.figure(figsize=(10, max(4, 0.32 * len(missing))))
plt.barh(missing["variable"], missing["pct_manquant"])
plt.xlabel("Valeurs manquantes (%)")
plt.title(f"{DATASET} — valeurs manquantes par métrique")
plt.tight_layout()
plt.show()
